# 🏥 General Health Query Chatbot
### Powered by Llama-3.1-8B via Hugging Face — Prompt Engineering Based

## Problem Statement
Build a chatbot that answers general health-related questions using a
Large Language Model (LLM) via API, with prompt engineering for friendly
responses and safety filters to avoid harmful medical advice.

## Tools Used
- Model: Meta Llama-3.1-8B-Instruct (via Hugging Face Router)
- API: Hugging Face Inference Providers API (free)
- Language: Python

## ⚠️ Disclaimer
This chatbot provides general health information only.
It is NOT a substitute for professional medical advice.
Always consult a qualified healthcare provider.

In [ ]:
%pip install openai -q

print("✅ Libraries loaded!")

In [ ]:
# ── Set your Hugging Face API token ──────────────────────────────────────────
# Replace with your actual token from huggingface.co/settings/tokens

from openai import OpenAI

HF_API_TOKEN = "PUT_YOUR_HF_TOKEN_HERE"

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=HF_API_TOKEN
)

print("✅ Hugging Face Router client ready!")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SAFETY FILTER
# Checks queries BEFORE sending to the LLM
# Two-layer safety: rule-based first, then LLM's own training second
# ─────────────────────────────────────────────────────────────────────────────

# Keywords that signal emergencies
EMERGENCY_KEYWORDS = [
    'chest pain', 'heart attack', "can't breathe", 'cant breathe',
    'difficulty breathing', 'stroke', 'unconscious', 'not breathing'
]

# Keywords that signal self-harm
SELF_HARM_KEYWORDS = [
    'kill myself', 'end my life', 'want to die',
    'suicide', 'self harm', 'hurt myself'
]

def safety_filter(query):
    """
    Pre-screens a user query before sending it to the LLM.
    
    Returns:
        (is_safe, message)
        is_safe = True  → send to LLM normally
        is_safe = False → return message directly, skip LLM
    """
    query_lower = query.lower()
    
    # Check for emergency symptoms
    for keyword in EMERGENCY_KEYWORDS:
        if keyword in query_lower:
            return False, (
                "🚨 EMERGENCY: The symptoms you described may be serious.\n"
                "Please call emergency services immediately:\n"
                "- Pakistan: 115 (Rescue)\n"
                "- UK: 999 | USA: 911\n"
                "Do NOT rely on a chatbot in an emergency!"
            )
    
    # Check for self-harm
    for keyword in SELF_HARM_KEYWORDS:
        if keyword in query_lower:
            return False, (
                "💙 It sounds like you're going through something very hard.\n"
                "Please reach out for support:\n"
                "- Pakistan Umang helpline: 0317-4288665\n"
                "- International: findahelpline.com\n"
                "You are not alone. Please talk to someone you trust."
            )
    
    # Query is safe to send to the LLM
    return True, None


print("✅ Safety filter ready!")
print(f"   Watching for {len(EMERGENCY_KEYWORDS)} emergency keywords")
print(f"   Watching for {len(SELF_HARM_KEYWORDS)} self-harm keywords")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CHATBOT CORE FUNCTION
# Sends query to Llama-3.1-8B via Hugging Face Router
# Maintains conversation history for multi-turn dialogue
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """Act like a helpful medical assistant who provides 
general health information. Follow these rules strictly:
- Use simple, friendly, and clear language
- Give general health information only — never diagnose anyone
- Never recommend specific prescription medications or dosages
- Always remind users to consult a real doctor for personal medical advice
- Keep answers concise and easy to understand
- Use bullet points where helpful"""

conversation_history = []

def ask_healthbot(user_question):
    """
    Main chatbot function.
    Steps:
      1. Run safety filter on the query
      2. Build messages list with conversation history
      3. Send to LLM via Hugging Face Router API
      4. Parse and return the response
      5. Save response to conversation history
    """
    print(f"\n👤 You: {user_question}")
    print("-" * 50)

    # Step 1: Safety check first
    is_safe, safety_message = safety_filter(user_question)
    if not is_safe:
        print(f"🛡️ SAFETY FILTER TRIGGERED:\n{safety_message}")
        return safety_message

    # Step 2: Build messages list with history
    try:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]

        for turn in conversation_history[-4:]:
            messages.append({"role": "user", "content": turn["user"]})
            messages.append({"role": "assistant", "content": turn["bot"]})

        messages.append({"role": "user", "content": user_question})

        # Step 3: Call Hugging Face API
        completion = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct:cerebras",
            messages=messages,
            max_tokens=400
        )

        # Step 4: Parse response
        bot_reply = completion.choices[0].message.content.strip()

        # Step 5: Save to history
        conversation_history.append({
            "user": user_question,
            "bot": bot_reply
        })

        print(f"🏥 HealthBot: {bot_reply}")
        print(f"\n⚕️  Remember: Always consult a real doctor for personal medical advice.")
        return bot_reply

    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return "An error occurred."


print("✅ Chatbot function ready!")

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO QUERIES — Including the two example queries from the task brief
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("        🏥 HEALTHBOT DEMO SESSION")
print("=" * 60)

# Query 1 — directly from the task brief
ask_healthbot("What causes a sore throat?");

        🏥 HEALTHBOT DEMO SESSION

👤 You: What causes a sore throat?
--------------------------------------------------
🏥 HealthBot: A sore throat can be caused by several things. Here are some common causes:

* **Viral infections**: The flu, common cold, and mononucleosis (mono) can all cause a sore throat.
* **Bacterial infections**: Strep throat (caused by group A streptococcus) is a type of bacterial infection that can cause a sore throat.
* **Irritants**: Smoking, pollution, and dry air can irritate the throat and cause discomfort.
* **Allergies**: Postnasal drip from allergies can flow down the back of the throat and cause a sore throat.
* **Acid reflux**: Gastroesophageal reflux disease (GERD) can cause stomach acid to flow up into the throat and irritate it.
* **Overuse or strain**: Shouting, singing, or talking loudly can put strain on the vocal cords and cause a sore throat.

If you're experiencing a sore throat, I recommend consulting a doctor to determine the cause and get p

In [6]:
# Query 2 — directly from the task brief
ask_healthbot("Is paracetamol safe for children?");


👤 You: Is paracetamol safe for children?
--------------------------------------------------
🏥 HealthBot: **Using paracetamol in children: safety guidelines**

Paracetamol (known as acetaminophen in some countries) can be safe for children when used correctly. Here are some guidelines:

* **Age recommendations:**
 + For children under 3 months: consult a doctor before giving paracetamol.
 + For children 3 months to 6 months: give a lower dose (10-20 mg per pound body weight) not exceeding 80 mg per dose.
 + For children 6 months and older: follow the normal dose instructions on the label.

**Important:** Always:

* Consult your doctor or pharmacist for advice on using paracetamol in your child.
* Follow the recommended dose and use the child-friendly formula of paracetamol.
* Never exceed the maximum dose or duration of use.
* Be aware of signs of overdose, such as vomiting, difficulty breathing, or drowsiness.

Remember, every child is different, and the safety of paracetamol use depe

In [7]:
# Query 3 — multi-turn test (bot remembers we discussed paracetamol)
ask_healthbot("What are the side effects of taking too much of it?");


👤 You: What are the side effects of taking too much of it?
--------------------------------------------------
🏥 HealthBot: **Risks of taking too much paracetamol:**

Taking too much paracetamol can lead to serious health problems, including:

* **Liver damage**: Too much paracetamol can cause liver failure, which may be fatal if not treated promptly.
* **Nausea and vomiting**: Taking too much paracetamol can cause stomach upset, leading to vomiting.
* **Abdominal pain**: High doses of paracetamol can cause stomach pain, cramps, or diarrhea.
* **Fatigue and weakness**: Excessive paracetamol use can make you feel tired, weak, or drowsy.
* **Dizziness or confusion**: Too much paracetamol can cause dizziness, confusion, or difficulty concentrating.

**Serious complications:**

In severe cases, taking too much paracetamol can lead to:

* **Liver failure**: This can cause jaundice (yellowing of the skin and eyes) and other symptoms.
* **Hepatic encephalopathy**: A life-threatening brain con

In [8]:
# Query 4 — general wellness
ask_healthbot("How much water should I drink every day?");


👤 You: How much water should I drink every day?
--------------------------------------------------
🏥 HealthBot: **Recommended water intake:**

The amount of water you should drink daily depends on various factors, such as:

* **Age**: Older adults need more water to stay hydrated.
* **Sex**: Pregnant or breastfeeding women need more water.
* **Weight**: More body weight means more water needs.
* **Activity level**: Athletes or individuals with physically demanding jobs need more water.
* **Climate**: Hot and humid environments require more water intake.

**General guidelines:**

* **Aim for 8-10 cups (64-80 ounces) per day**: This is a good starting point for most adults.
* **Drink half an ounce of water per pound of body weight**: For example, if you weigh 150 pounds, aim for 75 ounces of water per day.

Remember, these are general guidelines. Your individual needs may vary.

**Signs of proper hydration:**

* **Pale yellow urine**: This indicates you're drinking enough water.
* **Reg

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# SAFETY FILTER DEMO — These get BLOCKED before reaching the LLM
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("        🛡️ SAFETY FILTER DEMONSTRATIONS")
print("=" * 60)

# Should trigger emergency response
ask_healthbot("I have severe chest pain and can't breathe");

        🛡️ SAFETY FILTER DEMONSTRATIONS

👤 You: I have severe chest pain and can't breathe
--------------------------------------------------
🛡️ SAFETY FILTER TRIGGERED:
🚨 EMERGENCY: The symptoms you described may be serious.
Please call emergency services immediately:
- Pakistan: 115 (Rescue)
- UK: 999 | USA: 911
Do NOT rely on a chatbot in an emergency!


In [10]:
# Should trigger crisis/self-harm response
ask_healthbot("I want to hurt myself");


👤 You: I want to hurt myself
--------------------------------------------------
🛡️ SAFETY FILTER TRIGGERED:
💙 It sounds like you're going through something very hard.
Please reach out for support:
- Pakistan Umang helpline: 0317-4288665
- International: findahelpline.com
You are not alone. Please talk to someone you trust.


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# INTERACTIVE MODE — Type your own questions
# Type 'quit' to stop | Type 'reset' to clear conversation memory
# ─────────────────────────────────────────────────────────────────────────────

conversation_history.clear()

print("💬 Interactive Mode Started!")
print("Type 'quit' to exit | 'reset' to clear memory\n")

while True:
    try:
        user_input = input("👤 Your question: ").strip()
    except EOFError:
        break
    
    if not user_input:
        continue
    elif user_input.lower() == 'quit':
        print("👋 Thanks for using HealthBot. Stay healthy!")
        break
    elif user_input.lower() == 'reset':
        conversation_history.clear()
        print("🔄 Memory cleared. Fresh start!")
    else:
        ask_healthbot(user_input)

💬 Interactive Mode Started!
Type 'quit' to exit | 'reset' to clear memory


👤 You: my hand is numb
--------------------------------------------------
🏥 HealthBot: I'm here to help. Numbness in the hand can be uncomfortable and concerning. Here are some possible reasons:

• **Poor circulation**: Reduced blood flow to the hand can cause numbness.
• **Repetitive strain**: Activities like typing, gripping, or lifting may cause strain on the nerves or muscles in your hand.
• **Pressure on nerves**: Compression or rubbing of the nerves in your neck, arm, or hand can cause numbness.
• **Medical conditions**: Certain conditions, such as diabetes, multiple sclerosis, or peripheral neuropathy, can cause numbness in the hands.

What you can do:

• **Stay hydrated**: Drink plenty of water to help improve circulation.
• **Take breaks**: Rest your hands if you've been doing repetitive activities.
• **Stretch**: Gently stretch your hands, wrists, and arms.
• **Consult a doctor**: If the numbness pers

## Results & Final Insights

### What This Chatbot Demonstrates

| Skill | How It's Shown |
|-------|---------------|
| Prompt Engineering | System role + rules + tone defined in SYSTEM_PROMPT |
| LLM API Usage | Hugging Face Router API with Llama-3.1-8B-Instruct |
| Safety Handling | Keyword filter blocks harmful queries before reaching LLM |
| Conversational Agent | Full conversation history passed with each API request |

### Key Prompt Engineering Decisions
1. **Role first** — "Act like a helpful medical assistant" sets clear identity
2. **Explicit rules** — boundaries prevent the model from diagnosing
3. **Tone instruction** — "friendly and clear language" shapes response style
4. **Format guidance** — bullet points make responses more readable
5. **Disclaimer included** — every response reminds users to see a real doctor

### Safety Design
- Rule-based filter runs FIRST — fast and catches obvious dangerous cases
- LLM's own training acts as a second safety layer
- Emergency queries redirect to real emergency services immediately
- Self-harm queries redirect to crisis helplines with local Pakistan number

---
*Notebook by: Riyan Khan | Task 4 — Health Query Chatbot | DevelopersHub Internship*